# 01 — Data Loading & Exploratory Data Analysis

**Goal:** Load the raw Nasdaq market dataset, verify chronological integrity, and visualise the 7 macroeconomic / technical features alongside the price series.

**Modules used:** `src/data/load_data.py`, `src/evaluation/visiulation_artificial.py`

---

## 0 · Imports & path setup

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path so src.* imports work from the notebooks/ folder
ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from src.data.load_data import load_market_data

DATA_PATH = ROOT / 'data' / 'raw' / 'market_data_10y_enriched.csv'
DATE_COL  = 'Date'

print('Project root:', ROOT)
print('Data file   :', DATA_PATH)

---
## 1 · Load the raw CSV

`load_market_data` does three things:
1. Reads the CSV and parses the date column.
2. Sorts rows by date (ascending).
3. Asserts strict chronological order — raises `ValueError` if any duplicate or out-of-order date is found.

In [ ]:
df = load_market_data(str(DATA_PATH), date_col=DATE_COL)

print(f'Shape  : {df.shape}  ({df.shape[0]} trading days × {df.shape[1]} columns)')
print(f'Start  : {df[DATE_COL].iloc[0].date()}')
print(f'End    : {df[DATE_COL].iloc[-1].date()}')
print(f'Span   : {(df[DATE_COL].iloc[-1] - df[DATE_COL].iloc[0]).days} calendar days')
df.head()

---
## 2 · Schema & data types

In [ ]:
df.info()

---
## 3 · Descriptive statistics

In [ ]:
df.describe().T.style.format('{:.4f}')

---
## 4 · Missing value audit

Any missing values here will be handled by `apply_missing_value_policy` in the next notebook. We just count them now.

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

audit = pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct})
print(audit[audit.missing_count > 0].to_string() or 'No missing values found.')

---
## 5 · Feature descriptions

| Feature | Economic meaning |
|---|---|
| `VIX_Term_Structure` | Short-vs-long-term implied vol spread — contango = calm, backwardation = fear |
| `Yield_Curve` | 10Y – 2Y Treasury spread — inversion historically precedes recessions |
| `SKEW_Index` | Options-implied tail-risk; rising SKEW = market buying downside protection |
| `Risk_Appetite_Ratio` | Equity vs. safe-haven relative strength |
| `Crude_Oil` | Energy price as a global growth proxy |
| `VWAP_Deviation` | Distance of price from volume-weighted average — mean-reversion signal |
| `Volume_Momentum` | Trend in trading volume — confirms or contradicts price moves |

---
## 6 · Nasdaq price history

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df[DATE_COL], df['Nasdaq_Close'], linewidth=1.0, color='steelblue')

# Mark notable market events
events = {
    '2020-03-23': 'COVID low',
    '2022-01-04': '2022 peak',
    '2022-10-13': '2022 trough',
}
for date_str, label in events.items():
    ax.axvline(pd.Timestamp(date_str), color='red', linestyle='--', alpha=0.6)
    ax.text(pd.Timestamp(date_str), ax.get_ylim()[1] * 0.95, label,
            rotation=90, va='top', ha='right', fontsize=8, color='red')

ax.set_title('Nasdaq Composite — 10-Year Daily Close', fontsize=13)
ax.set_xlabel('Date')
ax.set_ylabel('Close Price')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()

---
## 7 · Macro feature time series

Each feature is plotted on its own axis to preserve scale.

In [ ]:
FEATURE_COLS = [
    'VIX_Term_Structure', 'Yield_Curve', 'SKEW_Index',
    'Risk_Appetite_Ratio', 'Crude_Oil', 'VWAP_Deviation', 'Volume_Momentum'
]

fig, axes = plt.subplots(len(FEATURE_COLS), 1, figsize=(14, 14), sharex=True)

colors = plt.cm.tab10.colors
for ax, col, color in zip(axes, FEATURE_COLS, colors):
    ax.plot(df[DATE_COL], df[col], linewidth=0.8, color=color)
    ax.set_ylabel(col, fontsize=8)
    ax.yaxis.set_tick_params(labelsize=7)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
axes[-1].set_xlabel('Date')
fig.suptitle('Macroeconomic & Technical Features — 10-Year History', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## 8 · Correlation matrix

In [ ]:
corr = df[FEATURE_COLS].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=12)
plt.tight_layout()
plt.show()

---
## 9 · Rolling 21-day volatility of Nasdaq returns

This gives us an intuition for the noise level that our classifier must overcome.

In [ ]:
daily_return = df['Nasdaq_Close'].pct_change()
rolling_vol  = daily_return.rolling(21).std() * (252 ** 0.5)  # annualised

fig, ax = plt.subplots(figsize=(14, 3))
ax.fill_between(df[DATE_COL], rolling_vol, alpha=0.5, color='tomato')
ax.set_title('Rolling 21-day Annualised Volatility', fontsize=12)
ax.set_ylabel('Annualised Vol')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()

---
## Summary

| Item | Value |
|---|---|
| Trading days loaded | `df.shape[0]` |
| Date range | ~10 years |
| Features | 7 macro / technical indicators |
| Chronological order | Verified ✓ |

**Next:** `02_preprocessing_and_labeling.ipynb` — handle NaN values, compute forward returns, and derive Bull/Bear labels.